# Word2Vec ile duygu analizi

### Alıştırma hedefleri:
- Word2Vec ile kelimeleri vektörlere dönüştürmek
- Word2vec tarafından verilen kelime temsilini RNN'ye beslemek için kullanmak

<hr>

▶️ Bu hücreyi çalıştırın ve kullandığınız 📚 [Gensim - Word2Vec](https://radimrehurek.com/gensim/auto_examples/index.html) sürümünün ≥ 4.0 olduğundan emin olun!

In [1]:
!pip freeze | grep gensim

gensim==4.3.3


# Veri


❓ **Soru** ❓ Öncelikle verileri yükleyelim. Fonksiyonda neler olduğunu anlamanıza gerek yok, burada önemi yok.

⚠️ **Uyarı** ⚠️ `load_data` fonksiyonunda `percentage_of_sentences` argümanı vardır. Bilgisayarınıza bağlı olarak, çok fazla cümle bilgisayarınızı yavaşlatabilir veya hatta dondurabilir - RAM'iniz taşabilir. Bu nedenle, **cümlelerin %10'uyla başlamalı** ve bilgisayarınızın bunu kaldırabildiğini kontrol etmelisiniz. Aksi takdirde, daha düşük bir sayı ile yeniden çalıştırın. 

⚠️ **DISCLAIMER** ⚠️ **_En büyüğü kimde_ (_who has the biggest_)(RAM) oyununu oynamaya gerek yok!** Buradaki amaç, modellerinizi hızlı bir şekilde çalıştırarak prototip oluşturmaktır. Gerçek hayatta bile, hızlı bir şekilde döngü ve hata ayıklama yapmak için verilerinizin bir alt kümesiyle başlamanız önerilir. Bu nedenle, yalnızca en iyi doğruluğu elde etmek istiyorsanız sayıyı artırın.

In [3]:
import numpy as np
import tensorflow_datasets as tfds
from tensorflow.keras.preprocessing.text import text_to_word_sequence

2026-04-04 00:00:02.391680: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-04 00:00:02.407117: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-04 00:00:02.559142: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-04 00:00:02.804939: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-04 00:00:02.981375: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registe

In [4]:
####################################################
### Verileri yüklemek için bu hücreyi çalıştırın ###
####################################################

import tensorflow_datasets as tfds
from tensorflow.keras.preprocessing.text import text_to_word_sequence

def load_data(percentage_of_sentences=None):
    train_data, test_data = tfds.load(name="imdb_reviews", split=["train", "test"], batch_size=-1, as_supervised=True)

    train_sentences, y_train = tfds.as_numpy(train_data)
    test_sentences, y_test = tfds.as_numpy(test_data)

    # Tüm verilerin yalnızca belirli bir yüzdesini alın
    if percentage_of_sentences is not None:
        assert(percentage_of_sentences> 0 and percentage_of_sentences<=100)

        len_train = int(percentage_of_sentences/100*len(train_sentences))
        train_sentences, y_train = train_sentences[:len_train], y_train[:len_train]

        len_test = int(percentage_of_sentences/100*len(test_sentences))
        test_sentences, y_test = test_sentences[:len_test], y_test[:len_test]

    X_train = [text_to_word_sequence(_.decode("utf-8")) for _ in train_sentences]
    X_test = [text_to_word_sequence(_.decode("utf-8")) for _ in test_sentences]

    return X_train, y_train, X_test, y_test

X_train, y_train, X_test, y_test = load_data(percentage_of_sentences=10)

2026-04-04 00:00:29.855245: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-04-04 00:00:29.859061: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2251] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Önceki alıştırmada, Word2vec temsilini eğittiniz ve bu Şekil'in ilk adımında gösterildiği gibi, tüm eğitim cümlelerinizi bir RNN'ye beslemek için dönüştürdünüz: 

<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/06-DL/NLP/word2vec_representation.png" alt="Word2Vec with RNN" width="400px" />



❓ **Soru** ❓ Burada, önceki alıştırmada yaptığınızın aynısını tekrar yapalım. İlk olarak, eğitim cümleniz üzerinde bir word2vec modeli (istediğiniz argümanlarla) eğitin. Bunu `word2vec` değişkenine kaydedin.

In [5]:
from gensim.models import Word2Vec

# Word2Vec modelini eğitiyoruz
# vector_size: Her kelimenin temsil edileceği vektör boyutu (örn: 100)
# window: Bir kelimenin sağındaki ve solundaki kaç kelimeye bakılacağı
# min_count: Bir kelimenin modelde yer alması için en az kaç kez geçmesi gerektiği
word2vec = Word2Vec(sentences=X_train, vector_size=100, window=5, min_count=2, workers=4)

Önceki alıştırmadaki işlevleri yeniden kullanarak, eğitim ve test verilerinizi RNN'ye girebileceğiniz bir biçime dönüştürelim.

❓ **Soru** ❓ Neler olduğunu anladığınızdan emin olmak için aşağıdaki işlevi okuyun ve çalıştırın.

In [6]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

# Bir cümleyi (kelime listesi) gömme uzayındaki kelimeleri temsil eden bir matrise dönüştürme işlevi
def embed_sentence(word2vec, sentence):
    embedded_sentence = []
    for word in sentence:
        if word in word2vec.wv:
            embedded_sentence.append(word2vec.wv[word])

    return np.array(embedded_sentence)

# Bir cümle listesini matris listesine dönüştüren işlev
def embedding(word2vec, sentences):
    embed = []

    for sentence in sentences:
        embedded_sentence = embed_sentence(word2vec, sentence)
        embed.append(embedded_sentence)

    return embed

# Eğitim ve test cümlelerini gömün(embed edin)
X_train_embed = embedding(word2vec, X_train)
X_test_embed = embedding(word2vec, X_test)


# Eğitim ve test gömülü cümleleri doldurun
X_train_pad = pad_sequences(X_train_embed, dtype='float32', padding='post', maxlen=200)
X_test_pad = pad_sequences(X_test_embed, dtype='float32', padding='post', maxlen=200)

☝️ Çalıştığından emin olmak için, `X_train_pad` ve `X_test_pad` için aşağıdakileri kontrol edelim:

- bunlar numpy dizileridir
- bunlar 3 boyutludur
- son boyut, word2vec gömme alanınızın boyutundadır (bunu `word2vec.wv.vector_size` ile elde edebilirsiniz
- ilk boyut, `X_train` ve `X_test` boyutundadır

✅ **İyi Uygulama** ✅ Bu tür testler oldukça önemlidir! Sadece bu alıştırmada değil, gerçek hayattaki uygulamalarda da. Hataları çok geç fark etmeyi ve bunların tüm not defterine yayılmasını önler.

In [7]:
# BENİ TEST ET
for X in [X_train_pad, X_test_pad]:
    assert type(X) == np.ndarray
    assert X.shape[-1] == word2vec.wv.vector_size


assert X_train_pad.shape[0] == len(X_train)
assert X_test_pad.shape[0] == len(X_test)

# Temel model

Kendi modelinizi test etmek için çok basit bir modele sahip olmak her zaman iyidir - çok basit bir algoritmadan daha iyi bir şey yaptığınızdan emin olmak için.

❓ **Soru** ❓ Temel doğruluk oranınız nedir? Bu durumda, temeliniz `y_train` içinde en çok bulunan etiketi tahmin etmek olabilir (tabii ki, veri kümesi dengeli ise, temel doğruluk oranı 1/n'dir; burada n, sınıfların sayısıdır - burada 2'dir).

In [8]:
# y_train içindeki 1'lerin (pozitif) oranına bakıyoruz
baseline_accuracy = np.mean(y_train)
print(f"Baseline Accuracy: {max(baseline_accuracy, 1 - baseline_accuracy):.4f}")

Baseline Accuracy: 0.5060


# Model

❓ **Soru** ❓ Aşağıdaki katmanlara sahip bir RNN yazın:
- bir `Masking` katmanı
- 20 birim ve `tanh` aktivasyon fonksiyonuna sahip bir `LSTM`
- 10 birimlik bir `Dense`
- görevinize bağlı bir çıktı katmanı

Ardından, modelinizi derleyin (en azından başlangıçta optimizer olarak `rmsprop` kullanmanızı öneririz).

In [9]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Masking, LSTM, Dense

def init_model():
    model = Sequential()
    
    # 1. Masking Katmanı: Dolgu (padding) olan 0 değerlerini işleme almaz
    model.add(Masking(mask_value=0.0, input_shape=(200, 100))) 
    
    # 2. LSTM Katmanı: 20 birim ve tanh aktivasyonu
    model.add(LSTM(20, activation='tanh'))
    
    # 3. Dense Katmanı: 10 birim
    model.add(Dense(10, activation='relu'))
    
    # 4. Çıktı Katmanı: İkili sınıflandırma (Pozitif/Negatif)
    model.add(Dense(1, activation='sigmoid'))

    # Modelin derlenmesi
    model.compile(loss='binary_crossentropy',
                  optimizer='rmsprop',
                  metrics=['accuracy'])
    
    return model

model = init_model()
model.summary()

/home/bariscan/.pyenv/versions/3.12.9/envs/workintech/lib/python3.12/site-packages/keras/src/layers/core/masking.py:48: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ masking (Masking)               │ (None, 200, 100)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 20)             │         9,680 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │           210 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            11 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,901 (38.68 KB)

 Trainable params: 9,901 (38.68 KB)

 Non-trainable params: 0 (0.00 B)

❓ **Soru** ❓ Modeli gömülü ve doldurulmuş verilerinize uyarlayın - erken durdurma kriterini unutmayın.

❗ **Not** ❗ Doğruluğunuz büyük ölçüde eğitim kümenize bağlı olacaktır. Burada, performansınızın temel modelin üzerinde olduğundan emin olun (bu, başlangıçtaki IMDB verilerinin yalnızca %20'sini yüklemiş olsanız bile geçerli olmalıdır).

In [10]:
from tensorflow.keras.callbacks import EarlyStopping

# Erken durdurma kriterini tanımlayalım
# 'val_loss' (doğrulama kaybı) artık iyileşmediğinde eğitimi durdurur
# 'restore_best_weights=True' ile en iyi performansı gösteren ağırlıklara geri döneriz
es = EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss')

# Modeli gömülü ve doldurulmuş (padded) verilerle eğitelim
history = model.fit(
    X_train_pad, 
    y_train, 
    validation_split=0.2, # Verinin %20'sini doğrulama için ayırıyoruz
    epochs=100, 
    batch_size=32, 
    callbacks=[es], 
    verbose=1
)

# Test seti üzerinde modeli değerlendirelim
results = model.evaluate(X_test_pad, y_test, verbose=0)
print(f'Test Seti Doğruluğu: {results[1]*100:.2f}%')

Epoch 1/100


2026-04-04 00:03:46.985045: E tensorflow/core/util/util.cc:131] oneDNN supports DT_BOOL only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


63/63 ━━━━━━━━━━━━━━━━━━━━ 9s 103ms/step - accuracy: 0.5180 - loss: 0.6934 - val_accuracy: 0.5440 - val_loss: 0.6872
Epoch 2/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 5s 80ms/step - accuracy: 0.5530 - loss: 0.6844 - val_accuracy: 0.5820 - val_loss: 0.6787
Epoch 3/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 6s 94ms/step - accuracy: 0.5855 - loss: 0.6736 - val_accuracy: 0.5920 - val_loss: 0.6706
Epoch 4/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 6s 87ms/step - accuracy: 0.6010 - loss: 0.6650 - val_accuracy: 0.5660 - val_loss: 0.6791
Epoch 5/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 6s 95ms/step - accuracy: 0.6075 - loss: 0.6550 - val_accuracy: 0.6200 - val_loss: 0.6517
Epoch 6/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 6s 93ms/step - accuracy: 0.6365 - loss: 0.6418 - val_accuracy: 0.6540 - val_loss: 0.6334
Epoch 7/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 6s 97ms/step - accuracy: 0.6505 - loss: 0.6317 - val_accuracy: 0.6400 - val_loss: 0.6358
Epoch 8/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 5s 79ms/step - accuracy: 0.6640 - loss: 0.6249 - val_accuracy: 0.6980 - val_

❓ **Soru** ❓ Test setinde modelinizi değerlendirin

In [11]:
# Modeli test seti (X_test_pad ve y_test) üzerinde değerlendirelim
results = model.evaluate(X_test_pad, y_test, verbose=1)

# Sonuçları ekrana yazdıralım
print(f'Test Kaybı (Loss): {results[0]:.4f}')
print(f'Test Doğruluğu (Accuracy): {results[1]*100:.2f}%')

79/79 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - accuracy: 0.6688 - loss: 0.6157
Test Kaybı (Loss): 0.6157
Test Doğruluğu (Accuracy): 66.88%


# Eğitimli Word2Vec - Transfer Öğrenimi

Doğruluğunuz, temel modelin üzerinde olsa da, oldukça düşük olabilir. Veri temizleme ve gömme kalitesini iyileştirme gibi bunu iyileştirmek için birçok seçenek vardır.

Burada veri temizleme stratejilerine girmeyeceğiz. Gömme kalitemizi iyileştirmeye çalışalım. Ancak, daha büyük bir metin kümesini yüklemek yerine, neden başkalarının öğrendiği gömme modelinden yararlanmayalım? Çünkü gömme modelinin kalitesi, yani kelimelerin yakınlığı, farklı görevlerden elde edilebilir. Transfer öğrenme tam olarak budur.

❓ **Soru** ❓ Bu sayede word2vec'te bulunan tüm farklı modelleri listeleyin: 

In [12]:
import gensim.downloader as api
print(list(api.info()['models'].keys()))

['fasttext-wiki-news-subwords-300', 'conceptnet-numberbatch-17-06-300', 'word2vec-ruscorpora-300', 'word2vec-google-news-300', 'glove-wiki-gigaword-50', 'glove-wiki-gigaword-100', 'glove-wiki-gigaword-200', 'glove-wiki-gigaword-300', 'glove-twitter-25', 'glove-twitter-50', 'glove-twitter-100', 'glove-twitter-200', '__testing_word2vec-matrix-synopsis']


ℹ️ Modellerin listesini ve boyutlarını [`gensim-data` deposunda](https://github.com/RaRe-Technologies/gensim-data#models) de bulabilirsiniz.

❓ **Soru** ❓ Önceden eğitilmiş word2vec gömme alanlarından birini yükleyin.

Bunu `api.load(seçtiğiniz model)` ile yapabilir ve `word2vec_transfer` içinde saklayabilirsiniz.

<details>
    <summary>💡 İpucu</summary>
    
`glove-wiki-gigaword-50` modeli, daha küçük olması (65 MB) nedeniyle başlangıç için iyi bir seçenektir.

</details>

In [13]:
import gensim.downloader as api

# Önceden eğitilmiş modeli yüklüyoruz
word2vec_transfer = api.load("glove-wiki-gigaword-50")

[==================================================] 100.0% 66.0/66.0MB downloaded


❓ **Soru** ❓ Kelime dağarcığının boyutunu ve aynı zamanda gömme alanının boyutunu kontrol edin.

In [14]:
# Kelime dağarcığı boyutu (toplam kaç farklı kelime bildiği)
vocab_size = len(word2vec_transfer)

# Gömme alanı (embedding) boyutu (her kelimenin kaç boyutlu bir vektör olduğu)
vector_size = word2vec_transfer.vector_size

print(f"Kelime Dağarcığı Boyutu: {vocab_size}")
print(f"Vektör Boyutu: {vector_size}")

Kelime Dağarcığı Boyutu: 400000
Vektör Boyutu: 50


❓ İlk soruda yaptığımız gibi, `X_train` ve `X_test`'i gömelim! (`embed_sentence_with_TF` işlevinde küçük bir fark var, ancak bu konuya girmeyeceğiz.)

In [15]:
# Bir cümleyi (kelime listesi) gömme uzayındaki kelimeleri temsil eden bir matrise dönüştürme işlevi
def embed_sentence_with_TF(word2vec, sentence):
    embedded_sentence = []
    for word in sentence:
        if word in word2vec:
            embedded_sentence.append(word2vec[word])

    return np.array(embedded_sentence)

# Bir cümle listesini matris listesine dönüştüren işlev
def embedding(word2vec, sentences):
    embed = []

    for sentence in sentences:
        embedded_sentence = embed_sentence_with_TF(word2vec, sentence)
        embed.append(embedded_sentence)

    return embed

# Eğitim ve test cümlelerini gömün (embed edin)
X_train_embed_2 = embedding(word2vec_transfer, X_train)
X_test_embed_2 = embedding(word2vec_transfer, X_test)

❓ **Soru** ❓  Sonuçlarınızı doldurmayı ve bunları `X_train_pad_2` ve `X_test_pad_2` içinde saklamayı unutmayın.

In [16]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Transfer learning ile oluşturulan gömülü cümleleri dolduruyoruz (padding)
X_train_pad_2 = pad_sequences(X_train_embed_2, dtype='float32', padding='post', maxlen=200)
X_test_pad_2 = pad_sequences(X_test_embed_2, dtype='float32', padding='post', maxlen=200)

❓ **Soru** ❓ Modeli yeniden başlatın ve yeni gömülü (ve doldurulmuş(padded)) verilerinize uyarlayın!  Test setinizde değerlendirin ve önceki doğruluğunuzla karşılaştırın.

❗ **Not** ❗ Buradaki eğitim biraz zaman alabilir. Sadece 10 dönem hesaplayabilir (bu **iyi** bir uygulama değildir, sadece çok uzun süre beklememek içindir) ve eğitim sürerken bir sonraki alıştırmaya geçebilir veya bir mola verebilirsiniz, muhtemelen bunu hak ettiniz ;)

In [19]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Masking, LSTM, Dense

# 1. Modeli 50 boyutlu (Glove) verisine göre SIFIRDAN tanımlıyoruz
model = Sequential()

# input_shape'in ikinci değerini 50 yapıyoruz
model.add(Masking(mask_value=0.0, input_shape=(200, 50))) 
model.add(LSTM(20, activation='tanh'))
model.add(Dense(10, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.compile(loss='binary_crossentropy',
              optimizer='rmsprop',
              metrics=['accuracy'])

# 2. Modeli yeni verilerle (X_train_pad_2) eğitiyoruz
model.fit(X_train_pad_2, y_train, 
          epochs=10, 
          batch_size=32, 
          validation_split=0.2)

Epoch 1/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 8s 102ms/step - accuracy: 0.5705 - loss: 0.6820 - val_accuracy: 0.5960 - val_loss: 0.6686
Epoch 2/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 6s 95ms/step - accuracy: 0.6120 - loss: 0.6632 - val_accuracy: 0.6440 - val_loss: 0.6434
Epoch 3/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 6s 95ms/step - accuracy: 0.6530 - loss: 0.6252 - val_accuracy: 0.6440 - val_loss: 0.6356
Epoch 4/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 4s 70ms/step - accuracy: 0.7055 - loss: 0.5837 - val_accuracy: 0.6560 - val_loss: 0.6297
Epoch 5/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 6s 91ms/step - accuracy: 0.7220 - loss: 0.5670 - val_accuracy: 0.6740 - val_loss: 0.6150
Epoch 6/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 6s 93ms/step - accuracy: 0.7355 - loss: 0.5441 - val_accuracy: 0.7200 - val_loss: 0.5582
Epoch 7/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 6s 93ms/step - accuracy: 0.7285 - loss: 0.5523 - val_accuracy: 0.7440 - val_loss: 0.5334
Epoch 8/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 6s 94ms/step - accuracy: 0.7320 - loss: 0.5329 - val_accuracy: 0.7260 - 

In [20]:
res = model.evaluate(X_test_pad_2, y_test, verbose=0)

print(f'The accuracy evaluated on the test set is of {res[1]*100:.3f}%')

The accuracy evaluated on the test set is of 74.120%


Yeni word2vec'iniz büyük bir metin kümesinde eğitildiğinden, çok sayıda kelimeyi temsil eder! Küçük veri kümenize kıyasla çok daha fazladır, özellikle de eğitim kümesinde belirli bir sayıdan fazla bulunmayan kelimeleri elediğiniz için. Bu nedenle, eğitim ve test kümenizde çok daha fazla gömülü kelime vardır, bu da her yinelemeyi öncekinden daha uzun hale getirir.